# NSRW E2 — Hugging Face FNO dataset physical audit

This notebook audits one pinned 2-D periodic vorticity trajectory. It does not train a model and does not prove a 3-D Navier–Stokes theorem. `claim_status` remains `UNVERIFIED`.

Before running, fill every `REQUIRED` field and materialize the bounded sample. The notebook intentionally fails closed when provenance is incomplete.

In [ ]:
import hashlib
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/NavierStokes-Research-Workbench')
SAMPLE_PATH = Path('/content/pinned_sample.npz')  # REQUIRED
SOURCE_SHA256 = 'REQUIRED'
REVISION = 'REQUIRED'
SOURCE_FILE = 'REQUIRED'
SAMPLE_ID = 'REQUIRED'
FORCING_SCOPE = 'REQUIRED'  # set only after checking the source metadata
if not SAMPLE_PATH.is_file() or 'REQUIRED' in {SOURCE_SHA256, REVISION, SOURCE_FILE, SAMPLE_ID, FORCING_SCOPE}:
    raise RuntimeError('E2 admission is held: pin revision, file, sample id, hash, and local sample')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))


In [ ]:
import numpy as np

from nsrw.external_experiments import ExternalSource, audit_vorticity_trajectory

actual_hash = hashlib.sha256(SAMPLE_PATH.read_bytes()).hexdigest()
if actual_hash.lower() != SOURCE_SHA256.lower():
    raise ValueError(f'sample hash mismatch: {actual_hash}')
with np.load(SAMPLE_PATH) as archive:
    if 'omega' not in archive:
        raise KeyError("pinned sample must contain an 'omega' array")
    omega = archive['omega']
source = ExternalSource(
    asset_id='abelsr1710/navier-stokes-2d-fno',
    source_url='https://huggingface.co/datasets/abelsr1710/navier-stokes-2d-fno',
    repository_id='abelsr1710/navier-stokes-2d-fno',
    revision=REVISION, file=SOURCE_FILE, source_file_sha256=SOURCE_SHA256,
    forcing_scope=FORCING_SCOPE,
    domain_lengths=(2.0 * np.pi, 2.0 * np.pi),  # REQUIRED: verify source coordinates
)
receipt = audit_vorticity_trajectory(omega, source=source, sample_id=SAMPLE_ID)
receipt


In [ ]:
output = Path('/content/nsrw-e2-receipt.json')
output.write_text(json.dumps(receipt, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(output, receipt['experiment_status'], receipt['claim_status'])
